In [4]:
import pandas as pd
import requests
from datetime import timedelta

In [ ]:
caminho_vendas = 'C:/Users/hiury/OneDrive/Área de Trabalho/IFCE/projeto_indicium/datasets/raw/vendas_2023_2024.csv'
df_vendas = pd.read_csv(caminho_vendas)

caminho_importacao = 'C:/Users/hiury/OneDrive/Área de Trabalho/IFCE/projeto_indicium/datasets/processed/df_importacao_final.csv'
df_importacao = pd.read_csv(caminho_importacao)

# Convertendo as datas
df_importacao['start_date'] = pd.to_datetime(df_importacao['start_date'], format='%d/%m/%Y')
df_vendas['sale_date'] = pd.to_datetime(df_vendas['sale_date'], format='mixed')

# Ordenando o dataframe por datas
df_importacao = df_importacao.sort_values('start_date')
df_vendas = df_vendas.sort_values('sale_date')

df_vendas.head(2)

,id,id_client,id_product,qtd,total,sale_date
4251,4294,7,44,5,51332.3,2023-01-01
8122,8211,14,67,3,257367.0,2023-01-01


In [24]:
# Merge_asof: para cada venda, busca o usd_price mais recente anterior ou igual à data da venda
df_merged = pd.merge_asof(
    df_vendas,
    df_importacao[['product_id', 'usd_price', 'start_date']],
    left_on='sale_date',
    right_on='start_date',
    left_by='id_product',
    right_by='product_id',
    direction='backward'
)



In [25]:
# Busca por câmbio do Banco Central
def buscar_cotacao_bcb(data):
    for i in range(5):
        data_tentativa = data - timedelta(days=i)
        data_str = data_tentativa.strftime('%m-%d-%Y')
        url = (
            f'https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/'
            f'CotacaoDolarDia(dataCotacao=@dataCotacao)'
            f'?@dataCotacao=%27{data_str}%27&$format=json&$top=1'
        )
        try:
            response = requests.get(url, timeout=10)
            dados = response.json()
            if dados['value']:
                return dados['value'][0]['cotacaoVenda']
        except Exception:
            continue
    return None

# Busca apenas datas únicas
print("Buscando cotações do BCB...")
datas_unicas = df_merged['sale_date'].dt.normalize().unique()
cotacoes = {data: buscar_cotacao_bcb(pd.Timestamp(data)) for data in datas_unicas}

# Cria dataframe de cotações e faz merge
df_cotacoes = pd.DataFrame(list(cotacoes.items()), columns=['sale_date_norm', 'cotacao_brl'])
df_merged['sale_date_norm'] = df_merged['sale_date'].dt.normalize()
df_merged = df_merged.merge(df_cotacoes, on='sale_date_norm', how='left')

Buscando cotações do BCB...


In [27]:
# Cálculo do custo e prejuízo

df_merged['custo_total_brl'] = df_merged['usd_price'] * df_merged['cotacao_brl'] * df_merged['qtd']
df_merged['prejuizo'] = round(df_merged['custo_total_brl'] - df_merged['total'],2)
df_merged['tem_prejuizo'] = df_merged['prejuizo'] > 0

df_merged.head(5)

,id,id_client,id_product,qtd,total,sale_date,product_id,usd_price,start_date,sale_date_norm,cotacao_brl,custo_total_brl,prejuizo,tem_prejuizo
0,4294,7,44,5,51332.3,2023-01-01,44,1963.02,2022-01-17,2023-01-01,5.2177,51212.247270,-120.05,False
1,8211,14,67,3,257367.0,2023-01-01,67,16720.73,2022-06-17,2023-01-01,5.2177,261731.258763,4364.26,True
2,4997,23,133,1,1893.0,2023-01-01,133,348.47,2022-07-06,2023-01-01,5.2177,1818.211919,-74.79,False
3,3131,28,130,13,53873.0,2023-01-01,130,749.89,2021-03-22,2023-01-01,5.2177,50865.113689,-3007.89,False
4,1230,17,91,4,512566.8,2023-01-01,91,26303.31,2022-03-16,2023-01-01,5.2177,548971.122348,36404.32,True


In [30]:
# Agregação por produto
df_agregado = df_merged.groupby('id_product').agg(
    receita_total  =('total', 'sum'),
    prejuizo_total =('prejuizo', lambda x: x[x > 0].sum()),
).reset_index()

df_agregado['percentual_perda'] = (
    df_agregado['prejuizo_total'] / df_agregado['receita_total'] * 100
).round(2)

df_agregado = df_agregado.sort_values('prejuizo_total', ascending=False)

df_agregado.head()

,id_product,receita_total,prejuizo_total,percentual_perda
71,72,63057815.65,39626286.10,62.84
82,83,44377440.00,19113308.95,43.07
70,71,81567066.65,6227298.65,7.63
73,74,59764356.15,6172882.33,10.33
54,55,61224375.00,5328679.56,8.70


In [35]:
df_merged.to_csv('C:/Users/hiury/OneDrive/Área de Trabalho/IFCE/projeto_indicium/datasets/output/vendas_com_prejuizo.csv', index=False)
df_agregado.to_csv('C:/Users/hiury/OneDrive/Área de Trabalho/IFCE/projeto_indicium/datasets/output/prejuizo_por_produto.csv', index=False)

print("Arquivos salvos em datasets/output/")

Arquivos salvos em datasets/output/


Para cada venda foi utilizada a cotação de venda do dólar do dia da transação, obtida via API pública do Banco Central do Brasil. Nos casos em que a data era fim de semana ou feriado  foi utilizada a cotação do último dia útil anterior, garantindo que nenhuma venda ficasse sem referência cambial.

Uma transação foi considerada com prejuízo quando o custo total em reais calculado multiplicando o usd_price na data da venda pela cotação do dia e pela quantidade vendida foi superior ao valor total registrado na venda.

A receita total inclui todas as vendas do produto, inclusive as lucrativas o percentual de perda reflete o impacto do prejuízo sobre o faturamento total do produto

In [7]:
'''
Para geração do dashboard
'''

df_prejuizo = pd.read_csv('C:/Users/hiury/OneDrive/Área de Trabalho/IFCE/projeto_indicium/datasets/output/prejuizo_por_produto.csv')
df_produto = pd.read_csv('C:/Users/hiury/OneDrive/Área de Trabalho/IFCE/projeto_indicium/datasets/processed/produtos_clean.csv')

df_prejuizo_final = df_prejuizo.merge(
    df_produto[['id_product', 'name']],
    left_on='id_product',
    right_on='id_product',
    how='left'
)


df_prejuizo_final[['name', 'prejuizo_total', 'percentual_perda']].to_csv(
    'C:/Users/hiury/OneDrive/Área de Trabalho/IFCE/projeto_indicium/datasets/output/prejuizo_com_nomes.csv',
    index=False,
    encoding='utf-8-sig'  
)